# R1B — Rust Checker Prompts (Token Without Concept)

**Purpose:** Generate prompts that contain a Rust keyword **token** but NOT the corresponding
syntactic **concept**. These are used to build token-checker circuits that distinguish
concept-encoding neurons from token-encoding neurons.

Mirrors 1B_checker_prompts.ipynb exactly, adapted for Rust syntax.

**Validation:** tree-sitter-rust replaces Python's `ast` module.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os

pkgs = ["transformers", "pyarrow", "pandas", "tqdm",
        "tree-sitter", "tree-sitter-rust", "numpy"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

N_TARGET = 50
OVERSHOOT_FACTOR = 3
run_name = "R1B_checker"
OUTPUT_FILE = f"{PREFIX}1B_checker_prompts.parquet"
MODEL_ID = "Qwen/Qwen2.5-Coder-7B"

LANG = "R"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = "/content/drive/MyDrive/DATA/New-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/New-Atlas"

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Data dir: {DATA_DIR}")
print(f"Target: {N_TARGET} prompts per object")

In [ ]:
# Cell 2 – Imports
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import tree_sitter_rust as ts_rust
from tree_sitter import Language, Parser

random.seed(42)
np.random.seed(42)

print("Imports OK")

In [ ]:
# Cell 3 – Load tokenizer (no model needed)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Tokenizer loaded: {MODEL_ID}")

In [ ]:
# Cell 4 – tree-sitter setup

RUST_LANGUAGE = Language(ts_rust.language())
ts_parser = Parser(RUST_LANGUAGE)


def parse_rust(code: str):
    return ts_parser.parse(bytes(code, "utf-8"))


def get_rust_node_types(code: str) -> set:
    tree = parse_rust(code)
    types = set()
    def walk(node):
        types.add(node.type)
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return types


def get_rust_identifiers(code: str) -> set:
    """Collect text of all identifier and type_identifier nodes."""
    tree = parse_rust(code)
    idents = set()
    def walk(node):
        if node.type in ("identifier", "type_identifier"):
            idents.add(node.text.decode("utf-8"))
        for child in node.children:
            walk(child)
    walk(tree.root_node)
    return idents


def has_errors(code: str) -> bool:
    return parse_rust(code).root_node.has_error


# Smoke test
test_code = 'fn main() { let x: i32 = 42; println!("{}", x); }'
print(f"Node types: {get_rust_node_types(test_code)}")
print(f"Identifiers: {get_rust_identifiers(test_code)}")
print(f"Has errors: {has_errors(test_code)}")
print("tree-sitter OK")

In [ ]:
# Cell 5 – RUST_KEYWORD_MAP

RUST_KEYWORD_MAP = {
    # ── Construct keywords ──
    "rust__For":       {"keyword": "for",      "forbidden_nodes": ["for_expression"]},
    "rust__While":     {"keyword": "while",    "forbidden_nodes": ["while_expression"]},
    "rust__Loop":      {"keyword": "loop",     "forbidden_nodes": ["loop_expression"]},
    "rust__If":        {"keyword": "if",       "forbidden_nodes": ["if_expression"]},
    "rust__Match":     {"keyword": "match",    "forbidden_nodes": ["match_expression"]},
    "rust__Fn":        {"keyword": "fn",       "forbidden_nodes": ["function_item"]},
    "rust__Let":       {"keyword": "let",      "forbidden_nodes": ["let_declaration"]},
    "rust__Struct":    {"keyword": "struct",   "forbidden_nodes": ["struct_item"]},
    "rust__Enum":      {"keyword": "enum",     "forbidden_nodes": ["enum_item"]},
    "rust__Impl":      {"keyword": "impl",     "forbidden_nodes": ["impl_item"]},
    "rust__Trait":     {"keyword": "trait",    "forbidden_nodes": ["trait_item"]},
    "rust__Use":       {"keyword": "use",      "forbidden_nodes": ["use_declaration"]},
    "rust__Mod":       {"keyword": "mod",      "forbidden_nodes": ["mod_item"]},
    "rust__Return":    {"keyword": "return",   "forbidden_nodes": ["return_expression"]},
    "rust__Break":     {"keyword": "break",    "forbidden_nodes": ["break_expression"]},
    "rust__Continue":  {"keyword": "continue", "forbidden_nodes": ["continue_expression"]},
    "rust__Const":     {"keyword": "const",    "forbidden_nodes": ["const_item"]},
    "rust__Static":    {"keyword": "static",   "forbidden_nodes": ["static_item"]},
    "rust__Async":     {"keyword": "async",    "forbidden_nodes": ["async_block"]},
    "rust__Await":     {"keyword": "await",    "forbidden_nodes": ["await_expression"]},
    "rust__Unsafe":    {"keyword": "unsafe",   "forbidden_nodes": ["unsafe_block"]},

    # ── Object keywords (types/traits used as identifiers) ──
    "rust__Vec":       {"keyword": "Vec",       "forbidden_nodes": [], "forbidden_idents": ["Vec"]},
    "rust__String":    {"keyword": "String",    "forbidden_nodes": [], "forbidden_idents": ["String"]},
    "rust__Option":    {"keyword": "Option",    "forbidden_nodes": [], "forbidden_idents": ["Option"]},
    "rust__Result":    {"keyword": "Result",    "forbidden_nodes": [], "forbidden_idents": ["Result"]},
    "rust__HashMap":   {"keyword": "HashMap",   "forbidden_nodes": [], "forbidden_idents": ["HashMap"]},
    "rust__Box":       {"keyword": "Box",       "forbidden_nodes": [], "forbidden_idents": ["Box"]},
    "rust__Arc":       {"keyword": "Arc",       "forbidden_nodes": [], "forbidden_idents": ["Arc"]},
    "rust__Rc":        {"keyword": "Rc",        "forbidden_nodes": [], "forbidden_idents": ["Rc"]},
    "rust__Clone":     {"keyword": "Clone",     "forbidden_nodes": [], "forbidden_idents": ["Clone"]},
    "rust__Iterator":  {"keyword": "Iterator",  "forbidden_nodes": [], "forbidden_idents": ["Iterator"]},
    "rust__Debug":     {"keyword": "Debug",     "forbidden_nodes": [], "forbidden_idents": ["Debug"]},
    "rust__Display":   {"keyword": "Display",   "forbidden_nodes": [], "forbidden_idents": ["Display"]},
    "rust__Default":   {"keyword": "Default",   "forbidden_nodes": [], "forbidden_idents": ["Default"]},
    "rust__From":      {"keyword": "From",      "forbidden_nodes": [], "forbidden_idents": ["From"]},
    "rust__Into":      {"keyword": "Into",      "forbidden_nodes": [], "forbidden_idents": ["Into"]},
    "rust__Mutex":     {"keyword": "Mutex",     "forbidden_nodes": [], "forbidden_idents": ["Mutex"]},
}

print(f"RUST_KEYWORD_MAP: {len(RUST_KEYWORD_MAP)} entries")
print(f"  Construct keywords: {sum(1 for k in RUST_KEYWORD_MAP if not any(k.endswith(t) for t in ('Vec','String','Option','Result','HashMap','Box','Arc','Rc','Clone','Iterator','Debug','Display','Default','From','Into','Mutex')))}")
print(f"  Object keywords: {sum(1 for k in RUST_KEYWORD_MAP if 'forbidden_idents' in RUST_KEYWORD_MAP[k])}")

In [ ]:
# Cell 6 – Word banks and identifier variants

RUST_IDENTIFIER_VARIANTS = {
    # Construct keywords
    "for":      ["format_string", "information", "formula_result", "before_update"],
    "while":    ["meanwhile_flag", "worthwhile", "awhile_counter"],
    "loop":     ["loopback_addr", "loop_str_ref", "unloop_count"],
    "if":       ["notification", "verification", "modification", "amplifier"],
    "match":    ["mismatch_count", "match_str_ref", "rematch_flag"],
    "fn":       ["config_val", "definition_str"],  # fn is hard to embed
    "let":      ["letter_count", "outlet_mode", "newsletter_flag"],
    "struct":   ["restructure", "constructor", "infrastructure"],
    "enum":     ["enumerate_items", "enumerated_val"],
    "impl":     ["implementation_id", "simplicity"],
    "trait":    ["portrait_id", "trait_str_ref"],
    "use":      ["user_name", "refuse_flag", "cause_str", "reuse_count"],
    "mod":      ["model_name", "modern_flag", "modify_count"],
    "return":   ["return_value_str", "unreturnable", "return_label"],
    "break":    ["breakdown_count", "breakpoint_flag", "unbreakable"],
    "continue": ["continuous_mode", "discontinue_flag", "continuum"],
    "const":    ["constant_val", "constraint_str", "construction"],
    "static":   ["statistic_val", "ecstatic_mode"],
    "async":    ["async_str_label", "asynchronous_flag"],
    "await":    ["await_str_label"],
    "unsafe":   ["unsafe_str_label"],
    # Object keywords
    "Vec":      ["vector_data", "vec_str_label"],
    "String":   ["string_data", "stringent_mode", "bowstring"],
    "Option":   ["optional_val", "option_str_ref"],
    "Result":   ["result_data", "resulting_flag"],
    "HashMap":  ["hashmap_str_ref"],
    "Box":      ["boxing_score", "mailbox_id", "sandbox_mode"],
    "Arc":      ["arcade_score", "arc_str_label", "architecture_id"],
    "Rc":       ["rc_str_label"],
    "Clone":    ["clone_str_label", "cyclone_flag"],
    "Iterator": ["iterator_str_label"],
    "Debug":    ["debug_str_label", "debugger_flag"],
    "Display":  ["display_str_label", "displayed_val"],
    "Default":  ["default_str_label", "defaulting_flag"],
    "From":     ["from_str_label", "frontline"],
    "Into":     ["into_str_label", "introduction"],
    "Mutex":    ["mutex_str_label"],
}

RUST_VAR_NAMES = [
    "data", "result_val", "value", "item", "record", "entry", "output",
    "signal", "status", "config", "mode_flag", "level", "target", "source",
    "handler", "token", "state", "buffer", "counter", "index",
    "offset", "block_id", "phase", "cycle", "frame", "chunk", "batch",
    "queue", "score", "label", "flag", "limit", "depth", "width_val",
]

RUST_STRING_WORDS = [
    "waiting", "ready", "starting", "completed", "processing",
    "running", "loading", "checking", "updating", "building",
    "reading", "writing", "sending", "receiving", "tracking",
    "reference", "documentation", "deployment", "production",
    "analysis", "operation", "execution", "validation", "review",
    "schedule", "planning", "progress", "workflow", "pipeline",
    "resource", "component", "structure", "algorithm", "protocol",
    "standard", "baseline", "threshold", "interval", "duration",
    "capacity", "frequency", "priority", "sequence", "pattern",
    "segment", "fragment", "section", "portion", "division",
]

RUST_CONTEXT_WRAPPERS = [
    "{snippet}",
    "let x: i32 = 1;\n{snippet}\nlet y: i32 = 2;",
    "let data: Vec<i32> = vec![];\n{snippet}\nlet result_val: Option<i32> = None;",
    "fn func() {{\n    {snippet}\n}}",
    "fn process() {{\n    {snippet}\n}}",
    "struct Obj;\nimpl Obj {{\n    fn method(&self) {{\n        {snippet}\n    }}\n}}",
]

print(f"Identifier variants: {len(RUST_IDENTIFIER_VARIANTS)} keywords")
print(f"Var names: {len(RUST_VAR_NAMES)}")
print(f"String words: {len(RUST_STRING_WORDS)}")
print(f"Context wrappers: {len(RUST_CONTEXT_WRAPPERS)}")

In [ ]:
# Cell 7 – Validation functions

def check_concept_absent_rust(code_string: str, obj_key: str) -> bool:
    """Verify that the syntactic concept is NOT present in the code."""
    info = RUST_KEYWORD_MAP[obj_key]
    node_types = get_rust_node_types(code_string)

    for forbidden in info["forbidden_nodes"]:
        if forbidden in node_types:
            return False

    if "forbidden_idents" in info:
        idents = get_rust_identifiers(code_string)
        for forbidden_ident in info["forbidden_idents"]:
            if forbidden_ident in idents:
                return False

    return True


def check_token_present(code_string: str, keyword: str, tokenizer) -> bool:
    """Verify that the keyword token appears in the tokenized code."""
    token_ids = tokenizer.encode(code_string, add_special_tokens=False)
    keyword_token_ids = tokenizer.encode(keyword, add_special_tokens=False)
    kw_len = len(keyword_token_ids)

    # Check for consecutive token match
    for i in range(len(token_ids) - kw_len + 1):
        if token_ids[i:i+kw_len] == keyword_token_ids:
            return True

    # Fallback: check individual decoded tokens
    for tid in token_ids:
        decoded = tokenizer.decode([tid]).strip()
        if decoded == keyword:
            return True

    return False


def validate_prompt_rust(code_string: str, obj_key: str) -> bool:
    """Full validation: concept absent AND token present AND no parse errors."""
    if has_errors(code_string):
        return False
    if not check_concept_absent_rust(code_string, obj_key):
        return False
    keyword = RUST_KEYWORD_MAP[obj_key]["keyword"]
    if not check_token_present(code_string, keyword, tokenizer):
        return False
    return True


print("Validation functions defined.")

In [ ]:
# Cell 8 – Generation categories + main loop

def generate_raw_prompts(keyword, n_per_category=10):
    """Generate raw candidate prompts across 5 categories."""
    candidates = []
    for _ in range(n_per_category):
        w1, w2, w3 = random.sample(RUST_STRING_WORDS, 3)
        v1, v2 = random.sample(RUST_VAR_NAMES, 2)
        wrapper = random.choice(RUST_CONTEXT_WRAPPERS)

        # Category A: String containing keyword
        snippet_a = random.choice([
            f'let {v1} = "waiting {keyword} {w1}";',
            f'let {v1} = "{w1} {keyword} {w2} reference";',
            f'let {v1} = "the {keyword} of {w1}";',
            f'let {v1} = "no {keyword} needed for {w1}";',
            f'let {v1} = "{w1} {keyword} {w2}";',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_a), "A_string"))

        # Category B: Comment containing keyword
        snippet_b = random.choice([
            f'let {v1}: i32 = 42;  // {keyword} {w1} see docs',
            f'// {keyword} {w1} {w2} reference\nlet {v1}: i32 = 10;',
            f'let {v1}: Vec<i32> = vec![];  // TODO {keyword} this later',
            f'// note: {keyword} not needed for {w1}\nlet {v1} = true;',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_b), "B_comment"))

        # Category C: Identifier containing keyword substring
        ident_variants = RUST_IDENTIFIER_VARIANTS.get(keyword, [])
        if ident_variants:
            ident = random.choice(ident_variants)
            snippet_c = random.choice([
                f'let {ident}: i32 = 42;',
                f'let {ident} = "{w1}";',
                f'let {ident} = true;',
            ])
            candidates.append((wrapper.replace("{snippet}", snippet_c), "C_identifier"))

        # Category D: Array of tuples (Rust analog of dict key)
        snippet_d = random.choice([
            f'let {v1} = [("{keyword}", true), ("{w1}", false)];',
            f'let {v1} = [("{w1}", 10), ("mode", 0)];\nlet {v2} = "{keyword}";',
            f'let {v1} = [("action", "{w1}"), ("{keyword}", "{w2}")];',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_d), "D_array_tuple"))

        # Category E: println! containing keyword
        snippet_e = random.choice([
            f'println!("{w1} {keyword} {w2}");',
            f'println!("starting {keyword} {w1}");',
            f'println!("{keyword} completed for {w1}");',
        ])
        candidates.append((wrapper.replace("{snippet}", snippet_e), "E_println"))

    return candidates


def generate_for_object(obj_key, target_n=N_TARGET):
    """Generate target_n validated prompts for one object."""
    keyword = RUST_KEYWORD_MAP[obj_key]["keyword"]
    raw = generate_raw_prompts(keyword, n_per_category=target_n * OVERSHOOT_FACTOR // 5)
    valid = [(t, c) for t, c in raw if validate_prompt_rust(t, obj_key)]

    by_cat = {}
    for text, cat in valid:
        by_cat.setdefault(cat, []).append(text)

    per_cat = target_n // 5
    selected = []
    for cat in ["A_string", "B_comment", "C_identifier", "D_array_tuple", "E_println"]:
        selected.extend(by_cat.get(cat, [])[:per_cat])

    if len(selected) < target_n:
        remaining = [t for t, c in valid if t not in selected]
        selected.extend(remaining[:target_n - len(selected)])

    return selected[:target_n]


# Main loop
all_prompts = {}
for obj_key in tqdm(sorted(RUST_KEYWORD_MAP.keys()), desc="Generating"):
    prompts = generate_for_object(obj_key)
    all_prompts[obj_key] = prompts
    status = "OK" if len(prompts) >= N_TARGET else f"SHORT ({len(prompts)})"
    print(f"  {obj_key:25s}: {len(prompts):3d} valid [{status}]")

total = sum(len(v) for v in all_prompts.values())
short = [k for k, v in all_prompts.items() if len(v) < N_TARGET]
print(f"\nTotal: {total} prompts across {len(all_prompts)} objects")
if short:
    print(f"WARNING: {len(short)} objects have < {N_TARGET} prompts: {short}")

In [ ]:
# Cell 9 – Save to parquet

records = []
for obj_key, prompts in sorted(all_prompts.items()):
    for i, text in enumerate(prompts):
        records.append({
            "object": obj_key,
            "keyword": RUST_KEYWORD_MAP[obj_key]["keyword"],
            "variation_id": i,
            "prompt_text": text,
        })

out_path = os.path.join(DATA_DIR, OUTPUT_FILE)
pd.DataFrame(records).to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Total rows: {len(records)}")

try:
    from google.colab import runtime
    runtime.unassign()
except (ImportError, AttributeError):
    print("Not running on Colab -- skipping unassign.")